# Optimal Graphical Indirect Sampling
## Two linked populations · prescribed inclusion probabilities · package GBFS

Edit the **single input cell**, then choose **Run All**. This notebook imports the
package implementation; the optimizer is the existing
`graphical_sampling.search.GreedyBestFirstSearch` with linked matching neighbors.

Every strip selects exactly $n$ distinct A units and $n$ distinct B units through
admissible links. Equal-mass exchanges preserve **every** $\lambda_{ik}$ and both
FIP vectors. Optimization uses auxiliary variables only; optional study variables
are used for simulation evaluation. GBFS is a heuristic, with no global optimality
guarantee. A design's **supported matchings** and the **candidate designs discovered
by search** are different objects; neither list enumerates all admissible designs.

Requires this feature branch installed with the package's normal dependencies,
plus Jupyter for interactive execution. Source-checkout execution also works from
this notebook's directory or a repository subdirectory.

In [ ]:
from pathlib import Path
import sys

# Prefer this checkout when launched inside it; otherwise use the installed package.
for directory in (Path.cwd(), *Path.cwd().parents):
    if (directory / 'graphical_sampling' / 'indirect').is_dir():
        sys.path.insert(0, str(directory))
        break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from graphical_sampling.indirect import (
    OGISInputs, LinkedDesign, optimize_gbfs, export_results,
    plot_links, plot_design, plot_results, variance_comparison, candidate_table,
)
from graphical_sampling.search import GreedyBestFirstSearch

### Your input — edit only the next cell

The default data use the manuscript's FIPs and links. The LP initializer may
choose a different feasible $\lambda$ and support than the manuscript example.
The exact 24-segment example is checked independently at the end.

In [ ]:
# Population identities are their row/column indices; labels are for display.
NAMES_A = ['A1', 'A2', 'A3', 'A4']
NAMES_B = ['B1', 'B2', 'B3', 'B4', 'B5']
pi_A = np.array([18, 12, 12, 6]) / 24
pi_B = np.array([13, 15, 7, 10, 3]) / 24
L = np.array([
    [1, 1, 0, 0, 0],
    [1, 0, 1, 1, 0],
    [0, 1, 0, 1, 0],
    [0, 0, 1, 0, 1],
])
x_A = np.array([3, 1, 3, .5])
x_B = np.ones(5)
y_A = np.array([5, 2, 6, 1])  # Optional simulation-only values; use None to omit.
y_B = np.array([4, 2, 7, 1, 3])

WEIGHT_A = .5
ITERATIONS = 150
PROPOSALS_PER_STEP = 35
EXCHANGE_FRACTION = .75
MAX_FRONTIER = 100
KEEP_BEST = 10
STAGNATION_LIMIT = 10
BACKTRACK_DEPTH = 50
SEED = 2026
SAVE_OUTPUTS = True
OUTPUT_DIR = 'ogis_outputs' 

### Feasibility and linked initialization

The transportation LP finds a nonnegative matrix on allowed links with row sums
$\pi_A$ and column sums $\pi_B$. Incompatible inputs raise a clear error before
optimization. Augmented Birkhoff decomposition then produces size-$n$ matchings.
Conditional assignment probabilities are $q_{ik}=\lambda_{ik}/\pi_i^A$.

In [ ]:
cfg = OGISInputs(pi_A, pi_B, L, x_A, x_B, y_A=y_A, y_B=y_B,
                 names_A=NAMES_A, names_B=NAMES_B)
initial_design = LinkedDesign.initialize(cfg)
print(f'A: {len(cfg.pi_A)} units · B: {len(cfg.pi_B)} units · n={cfg.n}')
print('Feasible fixed link masses:')
display(pd.DataFrame(initial_design.link_masses, index=cfg.names_A, columns=cfg.names_B))
print('Conditional assignment probabilities given A selection:')
display(pd.DataFrame(initial_design.conditional_probabilities,
                     index=cfg.names_A, columns=cfg.names_B))
fig, ax = plt.subplots(figsize=(10, 4))
plot_links(cfg, ax)
plt.show()

### Every supported initial sample pair

Each row is one matching and its probability under this initial design.
Probabilities sum to one; no A or B unit appears twice in a matching.

In [ ]:
initial_support = pd.DataFrame(initial_design.sample_table())
with pd.option_context('display.max_rows', None):
    display(initial_support)
fig, ax = plt.subplots(figsize=(11, 4))
plot_design(initial_design, ax)
plt.show()
print('Initial auxiliary NHT variances:', initial_design.nht_variance(cfg.x_A, cfg.x_B))

### Optimize using the package GBFS

For each matching, the NHT total is $\widehat X_A=\sum_{i\in s_A}x_i^A/\pi_i^A$
(and similarly for B). The exact variance is the probability-weighted squared
distance from the known auxiliary total. The objective is
$\alpha V_A/V_{A0}+(1-\alpha)V_B/V_{B0}$.
An exactly zero baseline uses unit scale, so later positive variance is penalized.
The package reservoir keeps its nine-decimal improvement policy.

In [ ]:
result = optimize_gbfs(
    initial_design, iterations=ITERATIONS, proposals=PROPOSALS_PER_STEP,
    alpha=WEIGHT_A, fraction=EXCHANGE_FRACTION, max_frontier=MAX_FRONTIER,
    seed=SEED, keep_best=KEEP_BEST, stagnation_limit=STAGNATION_LIMIT,
    backtrack_depth=BACKTRACK_DEPTH,
)
assert isinstance(result['search'], GreedyBestFirstSearch)
best_design = result['best']
assert best_design.validate()
print(f"Score: {result['initial_score']:.8f} → {result['best_score']:.8f}")
print(f"Unique discovered candidates: {result['generated_candidates']}")
print(f"Expanded keys retained by GBFS: {result['explored_states']}")
print('Auxiliary and available study-variable variance comparison:')
variance_summary = variance_comparison(result)
display(variance_summary)
fig = plot_results(result)
plt.show()

### Final support, both SIP matrices, and an actual matched sample

All final supported sample pairs are shown. SIP diagonal entries recover the
prescribed FIPs. Drawing selects one entire matching, preventing duplicates.

In [ ]:
best_support = pd.DataFrame(best_design.sample_table())
with pd.option_context('display.max_rows', None):
    display(best_support)
SA, SB = best_design.sip()
print('Final SIP matrix for A:')
display(pd.DataFrame(SA, index=cfg.names_A, columns=cfg.names_A))
print('Final SIP matrix for B:')
display(pd.DataFrame(SB, index=cfg.names_B, columns=cfg.names_B))
np.testing.assert_allclose(SA.diagonal(), cfg.pi_A, atol=1e-9, rtol=0)
np.testing.assert_allclose(SB.diagonal(), cfg.pi_B, atol=1e-9, rtol=0)
np.testing.assert_allclose(best_design.link_masses, initial_design.link_masses, atol=1e-9, rtol=0)
sampled_pairs = best_design.draw(seed=SEED)
print('One matched sample:')
display(pd.DataFrame([(cfg.names_A[i], cfg.names_B[k]) for i, k in sampled_pairs],
                     columns=['A unit', 'B unit']))

### Full discovered-candidate history and exports

The table contains **every unique candidate discovered**, in discovery order,
including the initial design. The gzip JSONL export stores the **complete support
distribution of each candidate**, with zero-based A/B indices and probabilities.
It does not enumerate all feasible designs. Initial/final support tables, both
initial/final SIP matrices, $\lambda$, $q$, variances, progress, metadata and PNG/PDF
figures are also exported. Large searches retain full candidate distributions
in memory and may be expensive.

In [ ]:
candidate_scores = candidate_table(result)
with pd.option_context('display.max_rows', None):
    display(candidate_scores)
display(pd.DataFrame(result['history']))
if SAVE_OUTPUTS:
    output_files = export_results(result, OUTPUT_DIR)
    print('Exported files:')
    for path in output_files:
        print(path.resolve())

### Independent exact check: manuscript 24-segment design

This fixed check does not use or override your input cell. It swaps complete
pairs A1→B1 in d8 and A2→B3 in d19, each of mass 1/24. The exact variances are
$(27/4,1823/325)$ before and $(73/12,10841/2275)$ after the exchange.

In [ ]:
paper_cfg = OGISInputs(
    np.array([18, 12, 12, 6])/24, np.array([13, 15, 7, 10, 3])/24,
    [[1,1,0,0,0], [1,0,1,1,0], [0,1,0,1,0], [0,0,1,0,1]],
    [3,1,3,.5], np.ones(5),
)
def P(*edges):
    return tuple(sorted((a-1, b-1) for a, b in edges))
paper_design = LinkedDesign(paper_cfg, {
    P((1,2),(2,1)):2/24, P((1,1),(2,3)):2/24,
    P((1,2),(2,4)):2/24, P((1,1),(3,2)):6/24,
    P((1,1),(3,4)):1/24, P((1,2),(3,4)):5/24,
    P((2,3),(4,5)):2/24, P((2,1),(4,5)):1/24,
    P((2,1),(4,3)):1/24, P((2,4),(4,3)):2/24,
})
after = paper_design.exchange(P((1,1),(3,2)), P((2,3),(4,5)), (0,0), (1,2), fraction=.5)
before_variances = paper_design.nht_variance(paper_cfg.x_A, paper_cfg.x_B)
after_variances = after.nht_variance(paper_cfg.x_A, paper_cfg.x_B)
np.testing.assert_allclose(before_variances, [27/4, 1823/325], atol=1e-12, rtol=0)
np.testing.assert_allclose(after_variances, [73/12, 10841/2275], atol=1e-12, rtol=0)
print('Manuscript check passed:', before_variances, '→', after_variances)